# 06_silver_aemet.ipynb — Limpieza AEMET Bronze → Silver

Este notebook procesa los JSON diarios de **AEMET** para generar:

```text
silver/meteo_hourly/source=AEMET/...
```

Aunque se guarda en `meteo_hourly` para mantener una tabla Silver común, AEMET se conserva como:

```text
temporal_resolution = daily
```

No se interpola a horario en Silver.

Variables principales:
- `temperature_air`, `temperature_min`, `temperature_max`
- `precipitation`
- `wind_speed`, `wind_direction`, `wind_gust`
- `pressure`, `pressure_min`, `pressure_max`
- `u10`, `v10` derivados si hay viento y dirección

Requiere que exista:

```text
silver/beach_geography/beach_geography.parquet
```

## Celda 0 — Montar Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [2]:
!pip -q install geopandas pyarrow shapely fiona tqdm

## Celda 2 — Imports, rutas y configuración

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

AEMET_DIR = BRONZE_DIR / "AEMET"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_METEO_DIR = SILVER_DIR / "meteo_hourly"
QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

for d in [OUT_METEO_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "AEMET"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")

# Para pruebas rápidas: pon 1 o 2. Para procesar todo, deja None.
MAX_STATIONS_FOR_TEST = None
MAX_FILES_PER_STATION_FOR_TEST = None

print("AEMET_DIR existe:", AEMET_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not AEMET_DIR.exists():
    raise FileNotFoundError(f"No existe AEMET_DIR: {AEMET_DIR}")

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

AEMET_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [4]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def infer_column(df, candidates):
    cols_norm = {normalize_col(col): col for col in df.columns}

    for candidate in candidates:
        candidate_norm = normalize_col(candidate)

        for col_norm, original_col in cols_norm.items():
            if candidate_norm == col_norm:
                return original_col

        for col_norm, original_col in cols_norm.items():
            if candidate_norm in col_norm:
                return original_col

    return None


def parse_aemet_compact_coordinate(value):
    """
    AEMET suele codificar coordenadas como:
    - latitud: DDMMSSN / DDMMSSS
    - longitud: DDDMMSSW / DDDMMSSE

    Ejemplos:
    280125N, 0152233W
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip().upper()
    s = s.replace(" ", "")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    hemi = None
    if s[-1:] in ["N", "S", "E", "W", "O"]:
        hemi = s[-1]
        digits = re.sub(r"\D", "", s[:-1])
    else:
        digits = re.sub(r"\D", "", s)

    if hemi in ["N", "S"] and len(digits) >= 6:
        deg = int(digits[:2])
        minutes = int(digits[2:4])
        seconds = int(digits[4:6])
        val = deg + minutes / 60 + seconds / 3600
        if hemi == "S":
            val = -val
        return float(val)

    if hemi in ["E", "W", "O"] and len(digits) >= 7:
        deg = int(digits[:3])
        minutes = int(digits[3:5])
        seconds = int(digits[5:7])
        val = deg + minutes / 60 + seconds / 3600
        if hemi in ["W", "O"]:
            val = -val
        return float(val)

    return np.nan


def parse_coordinate(value):
    """
    Convierte coordenadas decimales, DMS textual o formato compacto AEMET.
    """
    compact = parse_aemet_compact_coordinate(value)

    if not pd.isna(compact):
        return compact

    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        val = float(value)
        # Si viene como entero compacto sin hemisferio, no se puede saber si es lat/lon.
        return val

    s = str(value).strip().upper()
    s = s.replace(",", ".")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1
    if any(h in s for h in ["W", "O", "S"]):
        sign = -1
    if s.startswith("-"):
        sign = -1

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)

    if not nums:
        return np.nan

    try:
        if len(nums) >= 3 and ("º" in s or "°" in s or "'" in s or '"' in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            seconds = float(nums[2])
            val = deg + minutes / 60 + seconds / 3600
        elif len(nums) >= 2 and ("º" in s or "°" in s or "'" in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            val = deg + minutes / 60
        else:
            val = abs(float(nums[0])) if sign == -1 else float(nums[0])

        return sign * abs(val) if sign == -1 else val

    except Exception:
        return np.nan


def to_numeric_aemet(value, trace_to_zero=False):
    """
    Convierte números AEMET con coma decimal.
    - "Ip" o "Acum" se tratan como trazas si trace_to_zero=True.
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    if s == "":
        return np.nan

    s_upper = normalize_text(s)

    if s_upper in ["IP", "INAPRECIABLE", "TRAZA"]:
        return 0.0 if trace_to_zero else np.nan

    if s_upper in ["ACUM", "ACUMULADO"]:
        return np.nan

    s = s.replace(",", ".")
    s = re.sub(r"[^0-9eE+\-.]", "", s)

    if s in ["", "-", ".", "+"]:
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def series_numeric_aemet(series, trace_to_zero=False):
    return series.apply(lambda x: to_numeric_aemet(x, trace_to_zero=trace_to_zero))


def wind_components_from_speed_direction(speed, direction):
    """
    Dirección meteorológica: grados desde donde sopla el viento.
    """
    wd_rad = np.deg2rad(direction)
    u = -speed * np.sin(wd_rad)
    v = -speed * np.cos(wd_rad)
    return u, v


def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir):
    if df.empty:
        print("DataFrame vacío. No se guarda.")
        return

    df = df.copy()
    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )


def dataset_count_and_sample(path, source_name=SOURCE_NAME, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()

    return count, sample

## Celda 4 — Cargar `beach_geography`

In [5]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Leer metadata de estaciones AEMET

In [6]:
def read_csv_robust(path):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    seps = [";", ",", "\t"]

    last_error = None

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                if df.shape[1] > 1:
                    return df, {"encoding": enc, "sep": sep}
            except Exception as e:
                last_error = e

    raise last_error


station_csv_candidates = [
    AEMET_DIR / "aemet_estaciones_canarias.csv",
    AEMET_DIR / "aemet_estaciones_todas.csv",
]

station_csv_path = None

for p in station_csv_candidates:
    if p.exists():
        station_csv_path = p
        break

if station_csv_path is None:
    raise FileNotFoundError("No se encontró aemet_estaciones_canarias.csv ni aemet_estaciones_todas.csv.")

stations_raw, stations_read_info = read_csv_robust(station_csv_path)

print("CSV estaciones:", station_csv_path)
print("Lectura:", stations_read_info)
print("Shape:", stations_raw.shape)
print("Columnas:", stations_raw.columns.tolist())
display(stations_raw.head())

col_station_id = infer_column(stations_raw, ["indicativo", "idema", "station_id", "codigo", "cod"])
col_station_name = infer_column(stations_raw, ["nombre", "nombre estacion", "estacion"])
col_lat = infer_column(stations_raw, ["latitud", "lat", "latitude"])
col_lon = infer_column(stations_raw, ["longitud", "lon", "lng", "longitude"])
col_alt = infer_column(stations_raw, ["altitud", "alt", "altitude"])
col_prov = infer_column(stations_raw, ["provincia"])

detected_station_cols = {
    "station_id": col_station_id,
    "station_name": col_station_name,
    "lat": col_lat,
    "lon": col_lon,
    "altitude": col_alt,
    "provincia": col_prov,
}

print("Columnas detectadas:")
display(pd.DataFrame([detected_station_cols]))

if col_station_id is None:
    raise ValueError("No se pudo detectar columna de código/indicativo de estación.")

stations_meta = pd.DataFrame()
stations_meta["station_id"] = stations_raw[col_station_id].astype(str).str.strip()

if col_station_name is not None:
    stations_meta["station_name"] = stations_raw[col_station_name].astype(str).str.strip()
else:
    stations_meta["station_name"] = "AEMET_" + stations_meta["station_id"]

if col_lat is not None:
    stations_meta["lat"] = stations_raw[col_lat].apply(parse_coordinate)
else:
    stations_meta["lat"] = np.nan

if col_lon is not None:
    stations_meta["lon"] = stations_raw[col_lon].apply(parse_coordinate)
else:
    stations_meta["lon"] = np.nan

if col_alt is not None:
    stations_meta["altitude_m"] = stations_raw[col_alt].apply(lambda x: to_numeric_aemet(x))
else:
    stations_meta["altitude_m"] = np.nan

if col_prov is not None:
    stations_meta["provincia"] = stations_raw[col_prov].astype(str).str.strip()
else:
    stations_meta["provincia"] = np.nan

stations_meta = stations_meta.drop_duplicates(subset=["station_id"]).copy()

# Fallback aproximado para las 8 estaciones del proyecto, solo si falta coordenada.
AEMET_FALLBACK = {
    "C029O": {"station_name": "Gran Canaria / Las Palmas", "lat": 27.9319, "lon": -15.3866, "isla_hint": "Gran Canaria"},
    "C139E": {"station_name": "Tenerife Sur", "lat": 28.0445, "lon": -16.5725, "isla_hint": "Tenerife"},
    "C249I": {"station_name": "Lanzarote Aeropuerto", "lat": 28.9455, "lon": -13.6052, "isla_hint": "Lanzarote"},
    "C329B": {"station_name": "Fuerteventura Aeropuerto", "lat": 28.4527, "lon": -13.8638, "isla_hint": "Fuerteventura"},
    "C429I": {"station_name": "La Palma Aeropuerto", "lat": 28.6265, "lon": -17.7556, "isla_hint": "La Palma"},
    "C447A": {"station_name": "La Gomera Aeropuerto", "lat": 28.0296, "lon": -17.2146, "isla_hint": "La Gomera"},
    "C649I": {"station_name": "El Hierro Aeropuerto", "lat": 27.8148, "lon": -17.8871, "isla_hint": "El Hierro"},
    "C929I": {"station_name": "Tenerife Norte", "lat": 28.4827, "lon": -16.3415, "isla_hint": "Tenerife"},
}

stations_meta["coordinate_source"] = "csv_metadata"

for idx, row in stations_meta.iterrows():
    sid = row["station_id"]
    fb = AEMET_FALLBACK.get(sid)

    if fb is not None:
        if pd.isna(row["lat"]) or pd.isna(row["lon"]):
            stations_meta.loc[idx, "lat"] = fb["lat"]
            stations_meta.loc[idx, "lon"] = fb["lon"]
            stations_meta.loc[idx, "coordinate_source"] = "manual_fallback_project_station"

        if pd.isna(row["station_name"]) or str(row["station_name"]).strip() in ["", "nan"]:
            stations_meta.loc[idx, "station_name"] = fb["station_name"]

stations_meta["inside_bbox"] = (
    stations_meta["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & stations_meta["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("stations_meta:")
display(stations_meta)

if stations_meta["lat"].isna().any() or stations_meta["lon"].isna().any():
    print("AVISO: hay estaciones sin coordenadas:")
    display(stations_meta[stations_meta["lat"].isna() | stations_meta["lon"].isna()])

CSV estaciones: /content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/AEMET/aemet_estaciones_canarias.csv
Lectura: {'encoding': 'utf-8', 'sep': ','}
Shape: (104, 7)
Columnas: ['latitud', 'provincia', 'altitud', 'indicativo', 'nombre', 'indsinop', 'longitud']


,latitud,provincia,altitud,indicativo,nombre,indsinop,longitud
0,285811N,LAS PALMAS,376,C018J,TÍAS,NaN,134125W
1,285132N,LAS PALMAS,6,C019V,YAIZA PLAYA BLANCA,NaN,135006W
2,285707N,LAS PALMAS,14,C029O,LANZAROTE AEROPUERTO,60040.0,133601W
3,290837N,LAS PALMAS,277,C038N,HARÍA,NaN,132921W
4,290233N,LAS PALMAS,275,C048W,TINAJO,NaN,134051W


Columnas detectadas:


,station_id,station_name,lat,lon,altitude,provincia
0,indicativo,nombre,latitud,longitud,altitud,provincia


stations_meta:


,station_id,station_name,lat,lon,altitude_m,provincia,coordinate_source,inside_bbox
0,C018J,TÍAS,28.969722,-134125.0,376.0,LAS PALMAS,csv_metadata,False
1,C019V,YAIZA PLAYA BLANCA,28.858889,-135006.0,6.0,LAS PALMAS,csv_metadata,False
2,C029O,LANZAROTE AEROPUERTO,28.951944,-133601.0,14.0,LAS PALMAS,csv_metadata,False
3,C038N,HARÍA,29.143611,-132921.0,277.0,LAS PALMAS,csv_metadata,False
4,C048W,TINAJO,29.042500,-134051.0,275.0,LAS PALMAS,csv_metadata,False
...,...,...,...,...,...,...,...,...
99,C919K,TACORON-LAPILLAS-TORTUGA,27.665278,-180107.0,98.0,STA. CRUZ DE TENERIFE,csv_metadata,False
100,C925F,"SAN ANDRÉS, VALVERDE",27.768889,-175737.0,1070.0,STA. CRUZ DE TENERIFE,csv_metadata,False
101,C928I,VALVERDE,27.810833,-175510.0,670.0,STA. CRUZ DE TENERIFE,csv_metadata,False
102,C929I,HIERRO AEROPUERTO,27.818889,-175320.0,32.0,SANTA CRUZ DE TENERIFE,csv_metadata,False


5b parche

In [7]:
def parse_aemet_lat_lon(value, kind):
    """
    Parser específico para coordenadas AEMET.

    Soporta:
    - decimal normal: 28.951944, -13.6003
    - compacto con hemisferio: 285707N, 0133601W
    - compacto numérico: 285707, -133601
    """

    if pd.isna(value):
        return np.nan

    s = str(value).strip().upper()
    s = s.replace(",", ".")
    s = re.sub(r"\.0$", "", s)

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1

    if s.startswith("-"):
        sign = -1

    if any(h in s for h in ["S", "W", "O"]):
        sign = -1

    if any(h in s for h in ["N", "E"]):
        sign = 1

    # Primero intentamos decimal normal.
    try:
        numeric = float(re.sub(r"[^0-9eE+\-.]", "", s))

        if kind == "lat" and abs(numeric) <= 90:
            return numeric

        if kind == "lon" and abs(numeric) <= 180:
            # Canarias está al oeste. Si viene +13/+15 sin W, lo pasamos a negativo.
            if numeric > 0 and 13 <= numeric <= 19:
                return -numeric
            return numeric

    except Exception:
        pass

    digits = re.sub(r"\D", "", s)

    if kind == "lat":
        # Latitud AEMET: DDMMSS
        if len(digits) >= 6:
            deg = int(digits[:2])
            minutes = int(digits[2:4])
            seconds = int(digits[4:6])

            val = deg + minutes / 60 + seconds / 3600
            return sign * val

    if kind == "lon":
        # Longitud AEMET: DDDMMSS, pero a veces se pierde el cero inicial y queda DDMMSS.
        if len(digits) >= 7:
            deg = int(digits[:3])
            minutes = int(digits[3:5])
            seconds = int(digits[5:7])
        elif len(digits) >= 6:
            deg = int(digits[:2])
            minutes = int(digits[2:4])
            seconds = int(digits[4:6])
        else:
            return np.nan

        val = deg + minutes / 60 + seconds / 3600

        # Canarias: longitudes oeste.
        if sign > 0 and 13 <= val <= 19:
            sign = -1

        return sign * val

    return np.nan


def station_name_fallback_coords(station_name):
    """
    Fallback por nombre de estación si la coordenada sigue fuera de bbox.
    Solo se usa cuando el CSV no da coordenada válida.
    """

    n = normalize_text(station_name)

    if pd.isna(n):
        return None

    if "LANZAROTE" in n:
        return 28.9455, -13.6052

    if "FUERTEVENTURA" in n:
        return 28.4527, -13.8638

    if "GRAN CANARIA" in n or "GANDO" in n:
        return 27.9319, -15.3866

    if "TENERIFE SUR" in n or "REINA SOFIA" in n or "REINA SOFÍA" in n:
        return 28.0445, -16.5725

    if "TENERIFE NORTE" in n or "RODEOS" in n:
        return 28.4827, -16.3415

    if "LA PALMA" in n:
        return 28.6265, -17.7556

    if "LA GOMERA" in n:
        return 28.0296, -17.2146

    if "HIERRO" in n:
        return 27.8148, -17.8871

    return None


# Reconstruir stations_meta desde stations_raw usando el parser corregido.
stations_meta = pd.DataFrame()
stations_meta["station_id"] = stations_raw[col_station_id].astype(str).str.strip()

if col_station_name is not None:
    stations_meta["station_name"] = stations_raw[col_station_name].astype(str).str.strip()
else:
    stations_meta["station_name"] = "AEMET_" + stations_meta["station_id"]

if col_lat is not None:
    stations_meta["lat"] = stations_raw[col_lat].apply(lambda x: parse_aemet_lat_lon(x, "lat"))
else:
    stations_meta["lat"] = np.nan

if col_lon is not None:
    stations_meta["lon"] = stations_raw[col_lon].apply(lambda x: parse_aemet_lat_lon(x, "lon"))
else:
    stations_meta["lon"] = np.nan

if col_alt is not None:
    stations_meta["altitude_m"] = stations_raw[col_alt].apply(lambda x: to_numeric_aemet(x))
else:
    stations_meta["altitude_m"] = np.nan

if col_prov is not None:
    stations_meta["provincia"] = stations_raw[col_prov].astype(str).str.strip()
else:
    stations_meta["provincia"] = np.nan

stations_meta["coordinate_source"] = "csv_metadata_fixed_parser"

stations_meta = stations_meta.drop_duplicates(subset=["station_id"]).copy()

# Si alguna coordenada sigue fuera del bbox, intentar fallback por nombre.
for idx, row in stations_meta.iterrows():
    inside_bbox = (
        pd.notna(row["lat"])
        and pd.notna(row["lon"])
        and BBOX_CANARIAS["lat_min"] <= row["lat"] <= BBOX_CANARIAS["lat_max"]
        and BBOX_CANARIAS["lon_min"] <= row["lon"] <= BBOX_CANARIAS["lon_max"]
    )

    if not inside_bbox:
        fallback = station_name_fallback_coords(row["station_name"])

        if fallback is not None:
            stations_meta.loc[idx, "lat"] = fallback[0]
            stations_meta.loc[idx, "lon"] = fallback[1]
            stations_meta.loc[idx, "coordinate_source"] = "station_name_fallback"

stations_meta["inside_bbox"] = (
    stations_meta["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & stations_meta["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("stations_meta corregido:")
display(
    stations_meta[
        [
            "station_id",
            "station_name",
            "lat",
            "lon",
            "inside_bbox",
            "coordinate_source",
        ]
    ]
)

if "aemet_daily_raw" in globals():
    stations_with_data = sorted(aemet_daily_raw["station_id"].dropna().astype(str).unique())

    print("Estaciones con datos:")
    print(stations_with_data)

    display(
        stations_meta[
            stations_meta["station_id"].astype(str).isin(stations_with_data)
        ][
            [
                "station_id",
                "station_name",
                "lat",
                "lon",
                "inside_bbox",
                "coordinate_source",
            ]
        ]
    )

    bad_used = stations_meta[
        stations_meta["station_id"].astype(str).isin(stations_with_data)
        & ~stations_meta["inside_bbox"]
    ]

    if len(bad_used):
        display(bad_used)
        raise ValueError("Hay estaciones AEMET usadas con coordenadas fuera del bbox.")

print("Coordenadas AEMET corregidas.")

stations_meta corregido:


,station_id,station_name,lat,lon,inside_bbox,coordinate_source
0,C018J,TÍAS,28.969722,-13.690278,True,csv_metadata_fixed_parser
1,C019V,YAIZA PLAYA BLANCA,28.858889,-13.835000,True,csv_metadata_fixed_parser
2,C029O,LANZAROTE AEROPUERTO,28.951944,-13.600278,True,csv_metadata_fixed_parser
3,C038N,HARÍA,29.143611,-13.489167,True,csv_metadata_fixed_parser
4,C048W,TINAJO,29.042500,-13.680833,True,csv_metadata_fixed_parser
...,...,...,...,...,...,...
99,C919K,TACORON-LAPILLAS-TORTUGA,27.665278,-18.018611,True,csv_metadata_fixed_parser
100,C925F,"SAN ANDRÉS, VALVERDE",27.768889,-17.960278,True,csv_metadata_fixed_parser
101,C928I,VALVERDE,27.810833,-17.919444,True,csv_metadata_fixed_parser
102,C929I,HIERRO AEROPUERTO,27.818889,-17.888889,True,csv_metadata_fixed_parser


Coordenadas AEMET corregidas.


## Celda 6 — Localizar JSON diarios AEMET

In [8]:
station_dirs = sorted([p for p in AEMET_DIR.iterdir() if p.is_dir()])

if MAX_STATIONS_FOR_TEST is not None:
    station_dirs = station_dirs[:MAX_STATIONS_FOR_TEST]

rows = []

for station_dir in station_dirs:
    station_id = station_dir.name
    raw_files = sorted(station_dir.glob("aemet_daily_raw_*.json"))

    if MAX_FILES_PER_STATION_FOR_TEST is not None:
        raw_files = raw_files[:MAX_FILES_PER_STATION_FOR_TEST]

    for p in raw_files:
        rows.append(
            {
                "station_id": station_id,
                "path": str(p),
                "filename": p.name,
                "size_kb": round(p.stat().st_size / 1024, 2),
            }
        )

aemet_files_df = pd.DataFrame(rows)

print("Estaciones con carpeta:", len(station_dirs))
print("Archivos raw JSON encontrados:", len(aemet_files_df))

if len(aemet_files_df) == 0:
    raise FileNotFoundError("No se encontraron JSON raw diarios AEMET.")

display(aemet_files_df.head(20))
display(aemet_files_df["station_id"].value_counts().reset_index())

Estaciones con carpeta: 8
Archivos raw JSON encontrados: 162


,station_id,path,filename,size_kb
0,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2000_H1.json,90.52
1,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2000_H2.json,91.50
2,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2015_H1.json,106.29
3,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2015_H2.json,108.00
4,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2016_H1.json,106.45
5,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2016_H2.json,107.16
6,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2017_H1.json,106.20
7,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2017_H2.json,108.02
8,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2018_H1.json,105.86
9,C029O,/content/drive/MyDrive/AI Projects/DeepWave Ca...,aemet_daily_raw_C029O_2018_H2.json,107.57


,station_id,count
0,C029O,24
1,C139E,22
2,C249I,22
3,C429I,22
4,C649I,22
5,C447A,22
6,C929I,22
7,C329B,6


## Celda 7 — Lectura robusta de JSON AEMET

In [9]:
def load_json_robust(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read().strip()

    if text == "":
        return []

    try:
        obj = json.loads(text)
    except Exception:
        # Intento latin1 si hubiera problemas raros.
        with open(path, "r", encoding="latin1", errors="replace") as f:
            obj = json.loads(f.read())

    # AEMET puede devolver una lista directa.
    if isinstance(obj, list):
        return obj

    # O un dict con datos.
    if isinstance(obj, dict):
        for key in ["datos", "data", "items", "observaciones", "result"]:
            if key in obj and isinstance(obj[key], list):
                return obj[key]

        # A veces puede venir un string JSON dentro de un campo.
        for key in ["datos", "data"]:
            if key in obj and isinstance(obj[key], str):
                try:
                    inner = json.loads(obj[key])
                    if isinstance(inner, list):
                        return inner
                except Exception:
                    pass

        # Si el dict parece un único registro.
        if any(k in obj for k in ["fecha", "indicativo", "tmed", "prec"]):
            return [obj]

    return []


def read_aemet_file(path, station_id_from_folder):
    records = load_json_robust(path)

    if not records:
        return pd.DataFrame(), {
            "filename": Path(path).name,
            "station_id": station_id_from_folder,
            "rows": 0,
            "status": "empty_or_unrecognized",
        }

    df = pd.DataFrame(records)
    df.columns = [str(c).strip() for c in df.columns]

    if "indicativo" not in df.columns:
        df["indicativo"] = station_id_from_folder

    # Algunos registros pueden traer espacios.
    df["indicativo"] = df["indicativo"].astype(str).str.strip()

    summary = {
        "filename": Path(path).name,
        "station_id": station_id_from_folder,
        "rows": len(df),
        "status": "ok",
        "columns": json.dumps(df.columns.tolist(), ensure_ascii=False),
    }

    return df, summary

## Celda 8 — Estandarización diaria AEMET

In [10]:
def find_first_column(df, candidates):
    return infer_column(df, candidates)


def normalize_aemet_daily_df(df_raw, station_id_from_folder, filename):
    df = df_raw.copy()

    if df.empty:
        return pd.DataFrame()

    col_fecha = find_first_column(df, ["fecha", "date", "dia"])
    if col_fecha is None:
        raise ValueError(f"{filename}: no se pudo detectar columna fecha.")

    timestamp = pd.to_datetime(df[col_fecha], errors="coerce", utc=True)

    # Si venía como fecha sin hora, queda a 00:00 UTC.
    out = pd.DataFrame()
    out["timestamp"] = timestamp
    out["date"] = out["timestamp"].dt.date

    col_station = find_first_column(df, ["indicativo", "station_id", "idema"])
    if col_station is not None:
        out["station_id"] = df[col_station].astype(str).str.strip()
    else:
        out["station_id"] = station_id_from_folder

    col_nombre = find_first_column(df, ["nombre", "estacion"])
    if col_nombre is not None:
        out["station_name_raw"] = df[col_nombre].astype(str).str.strip()
    else:
        out["station_name_raw"] = np.nan

    # Temperaturas.
    col_tmed = find_first_column(df, ["tmed", "temperatura media", "temp_media"])
    col_tmin = find_first_column(df, ["tmin", "temperatura minima", "temperatura mínima", "temp_min"])
    col_tmax = find_first_column(df, ["tmax", "temperatura maxima", "temperatura máxima", "temp_max"])

    out["temperature_air"] = series_numeric_aemet(df[col_tmed]) if col_tmed is not None else np.nan
    out["temperature_min"] = series_numeric_aemet(df[col_tmin]) if col_tmin is not None else np.nan
    out["temperature_max"] = series_numeric_aemet(df[col_tmax]) if col_tmax is not None else np.nan

    # Precipitación.
    col_prec = find_first_column(df, ["prec", "precipitacion", "precipitación", "precipitation"])
    if col_prec is not None:
        out["precipitation"] = series_numeric_aemet(df[col_prec], trace_to_zero=True)
        out["precipitation_is_trace"] = df[col_prec].astype(str).map(lambda x: normalize_text(x) in ["IP", "INAPRECIABLE", "TRAZA"])
    else:
        out["precipitation"] = np.nan
        out["precipitation_is_trace"] = False

    # Viento.
    col_velmedia = find_first_column(df, ["velmedia", "vmedia", "velocidad media", "wind_speed"])
    col_racha = find_first_column(df, ["racha", "vmax", "wind_gust"])
    col_dir = find_first_column(df, ["dir", "viento_dir", "direccion viento", "dirección viento", "wind_direction"])

    out["wind_speed"] = series_numeric_aemet(df[col_velmedia]) if col_velmedia is not None else np.nan
    out["wind_gust"] = series_numeric_aemet(df[col_racha]) if col_racha is not None else np.nan

    if col_dir is not None:
        direction = series_numeric_aemet(df[col_dir])
        # AEMET diario suele codificar dirección en decenas de grados: 01..36.
        direction = np.where(
            pd.Series(direction).between(0, 36),
            pd.Series(direction) * 10,
            pd.Series(direction),
        )
        out["wind_direction"] = pd.Series(direction, index=out.index).astype(float) % 360
    else:
        out["wind_direction"] = np.nan

    # Presión.
    col_presmax = find_first_column(df, ["presmax", "presMax", "presion maxima", "presión máxima", "pressure_max"])
    col_presmin = find_first_column(df, ["presmin", "presMin", "presion minima", "presión mínima", "pressure_min"])

    out["pressure_max"] = series_numeric_aemet(df[col_presmax]) if col_presmax is not None else np.nan
    out["pressure_min"] = series_numeric_aemet(df[col_presmin]) if col_presmin is not None else np.nan
    out["pressure"] = out[["pressure_max", "pressure_min"]].mean(axis=1)

    # Humedad si apareciera.
    col_hum = find_first_column(df, ["humedad", "hr", "humidity", "hum_relativa"])
    out["humidity"] = series_numeric_aemet(df[col_hum]) if col_hum is not None else np.nan

    # Derivar u/v si hay velocidad y dirección.
    if "wind_speed" in out.columns and "wind_direction" in out.columns:
        u, v = wind_components_from_speed_direction(out["wind_speed"], out["wind_direction"])
        out["u10"] = u
        out["v10"] = v
    else:
        out["u10"] = np.nan
        out["v10"] = np.nan

    out["source_file"] = filename
    out["temporal_resolution"] = "daily"
    out["source"] = SOURCE_NAME

    out = out.dropna(subset=["timestamp", "station_id"]).copy()

    invalid_time = ~out["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)
    if invalid_time.any():
        raise ValueError(
            f"{filename}: timestamps fuera de rango "
            f"{out.loc[invalid_time, 'timestamp'].min()} - {out.loc[invalid_time, 'timestamp'].max()}"
        )

    return out

## Celda 9 — Procesar todos los JSON diarios

In [11]:
processed_parts = []
file_summaries = []
read_errors = []

for _, row in tqdm(aemet_files_df.iterrows(), total=len(aemet_files_df), desc="Procesando AEMET"):
    path = Path(row["path"])
    station_id = row["station_id"]

    try:
        raw_df, summary = read_aemet_file(path, station_id)
        summary["path"] = str(path)
        summary["raw_rows"] = len(raw_df)

        if raw_df.empty:
            summary["standardized_rows"] = 0
            file_summaries.append(summary)
            continue

        std_df = normalize_aemet_daily_df(raw_df, station_id, path.name)

        summary["standardized_rows"] = len(std_df)
        summary["timestamp_min"] = std_df["timestamp"].min() if len(std_df) else pd.NaT
        summary["timestamp_max"] = std_df["timestamp"].max() if len(std_df) else pd.NaT

        processed_parts.append(std_df)
        file_summaries.append(summary)

    except Exception as e:
        read_errors.append(
            {
                "filename": path.name,
                "station_id": station_id,
                "path": str(path),
                "error": repr(e),
            }
        )

file_summary_df = pd.DataFrame(file_summaries)
read_errors_df = pd.DataFrame(read_errors)

print("Archivos procesados correctamente:", len(file_summary_df))
print("Errores de lectura:", len(read_errors_df))

display(file_summary_df.head())
display(read_errors_df)

file_summary_df.to_csv(QC_DIR / "quality_aemet_file_summary.csv", index=False)
read_errors_df.to_csv(QC_DIR / "quality_aemet_read_errors.csv", index=False)

if len(read_errors_df):
    raise ValueError("Hay errores leyendo AEMET. Revisar quality_aemet_read_errors.csv.")

if not processed_parts:
    raise ValueError("No se procesó ningún dato diario AEMET.")

aemet_daily_raw = pd.concat(processed_parts, ignore_index=True)

print("aemet_daily_raw shape:", aemet_daily_raw.shape)
display(aemet_daily_raw.head())

Procesando AEMET:   0%|          | 0/162 [00:00<?, ?it/s]

Archivos procesados correctamente: 162
Errores de lectura: 0


,filename,station_id,rows,status,columns,path,raw_rows,standardized_rows,timestamp_min,timestamp_max
0,aemet_daily_raw_C029O_2000_H1.json,C029O,182,ok,"[""fecha"", ""indicativo"", ""nombre"", ""provincia"",...",/content/drive/MyDrive/AI Projects/DeepWave Ca...,182,182,2000-01-01 00:00:00+00:00,2000-06-30 00:00:00+00:00
1,aemet_daily_raw_C029O_2000_H2.json,C029O,184,ok,"[""fecha"", ""indicativo"", ""nombre"", ""provincia"",...",/content/drive/MyDrive/AI Projects/DeepWave Ca...,184,184,2000-07-01 00:00:00+00:00,2000-12-31 00:00:00+00:00
2,aemet_daily_raw_C029O_2015_H1.json,C029O,181,ok,"[""fecha"", ""indicativo"", ""nombre"", ""provincia"",...",/content/drive/MyDrive/AI Projects/DeepWave Ca...,181,181,2015-01-01 00:00:00+00:00,2015-06-30 00:00:00+00:00
3,aemet_daily_raw_C029O_2015_H2.json,C029O,184,ok,"[""fecha"", ""indicativo"", ""nombre"", ""provincia"",...",/content/drive/MyDrive/AI Projects/DeepWave Ca...,184,184,2015-07-01 00:00:00+00:00,2015-12-31 00:00:00+00:00
4,aemet_daily_raw_C029O_2016_H1.json,C029O,182,ok,"[""fecha"", ""indicativo"", ""nombre"", ""provincia"",...",/content/drive/MyDrive/AI Projects/DeepWave Ca...,182,182,2016-01-01 00:00:00+00:00,2016-06-30 00:00:00+00:00


""


aemet_daily_raw shape: (27578, 21)


,timestamp,date,station_id,station_name_raw,temperature_air,temperature_min,temperature_max,precipitation,precipitation_is_trace,wind_speed,...,wind_direction,pressure_max,pressure_min,pressure,humidity,u10,v10,source_file,temporal_resolution,source
0,2000-01-01 00:00:00+00:00,2000-01-01,C029O,LANZAROTE AEROPUERTO,17.0,14.4,19.6,0.0,False,5.3,...,60.0,1021.6,1018.0,1019.8,89.0,-4.589935,-2.650000,aemet_daily_raw_C029O_2000_H1.json,daily,AEMET
1,2000-01-02 00:00:00+00:00,2000-01-02,C029O,LANZAROTE AEROPUERTO,16.2,13.5,19.0,0.0,False,4.4,...,120.0,1024.2,1020.4,1022.3,73.0,-3.810512,2.200000,aemet_daily_raw_C029O_2000_H1.json,daily,AEMET
2,2000-01-03 00:00:00+00:00,2000-01-03,C029O,LANZAROTE AEROPUERTO,16.5,13.4,19.6,0.0,False,4.4,...,70.0,1025.0,1021.2,1023.1,81.0,-4.134648,-1.504889,aemet_daily_raw_C029O_2000_H1.json,daily,AEMET
3,2000-01-04 00:00:00+00:00,2000-01-04,C029O,LANZAROTE AEROPUERTO,16.6,13.8,19.4,0.0,False,3.6,...,20.0,1024.2,1020.8,1022.5,85.0,-1.231273,-3.382893,aemet_daily_raw_C029O_2000_H1.json,daily,AEMET
4,2000-01-05 00:00:00+00:00,2000-01-05,C029O,LANZAROTE AEROPUERTO,16.3,13.9,18.7,0.0,False,4.2,...,170.0,1022.4,1019.2,1020.8,65.0,-0.729322,4.136193,aemet_daily_raw_C029O_2000_H1.json,daily,AEMET


## Celda 10 — Asignar estaciones AEMET a zonas

In [12]:
# Quedarse solo con estaciones que tienen datos.
stations_with_data = sorted(aemet_daily_raw["station_id"].dropna().astype(str).unique())

station_meta_used = stations_meta[
    stations_meta["station_id"].astype(str).isin(stations_with_data)
].copy()

# Si hay datos de una estación sin metadata, añadir fila con NaN para fallar de forma clara.
missing_meta_ids = sorted(set(stations_with_data) - set(station_meta_used["station_id"].astype(str)))

if missing_meta_ids:
    print("AVISO: estaciones con datos sin metadata:", missing_meta_ids)
    extra = pd.DataFrame(
        {
            "station_id": missing_meta_ids,
            "station_name": [f"AEMET_{x}" for x in missing_meta_ids],
            "lat": np.nan,
            "lon": np.nan,
            "altitude_m": np.nan,
            "provincia": np.nan,
            "coordinate_source": "missing_metadata",
        }
    )
    station_meta_used = pd.concat([station_meta_used, extra], ignore_index=True)

if station_meta_used["lat"].isna().any() or station_meta_used["lon"].isna().any():
    display(station_meta_used[station_meta_used["lat"].isna() | station_meta_used["lon"].isna()])
    raise ValueError("Hay estaciones AEMET con datos pero sin coordenadas.")

gdf_stations = gpd.GeoDataFrame(
    station_meta_used.copy(),
    geometry=gpd.points_from_xy(station_meta_used["lon"], station_meta_used["lat"]),
    crs="EPSG:4326",
)

gdf_stations_m = gdf_stations.to_crs("EPSG:3857")

nearest = gpd.sjoin_nearest(
    gdf_stations_m,
    gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
    how="left",
    distance_col="distance_to_zona_m",
)

nearest = (
    nearest
    .sort_values("distance_to_zona_m")
    .groupby("station_id", as_index=False)
    .first()
)

station_zone = pd.DataFrame(nearest.drop(columns="geometry", errors="ignore"))
station_zone["distance_to_zona_km"] = station_zone["distance_to_zona_m"] / 1000

station_zone = station_zone[
    [
        "station_id",
        "station_name",
        "lat",
        "lon",
        "altitude_m",
        "provincia",
        "coordinate_source",
        "zona_id",
        "nombre_zona",
        "isla",
        "municipio",
        "distance_to_zona_km",
    ]
].copy()

display(station_zone)

station_zone.to_csv(META_DIR / "aemet_station_to_zone.csv", index=False)

print("Distancia estación AEMET → zona más cercana, km:")
display(station_zone["distance_to_zona_km"].describe())

,station_id,station_name,lat,lon,altitude_m,provincia,coordinate_source,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,C029O,LANZAROTE AEROPUERTO,28.951944,-13.600278,14.0,LAS PALMAS,csv_metadata_fixed_parser,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041
1,C139E,LA PALMA AEROPUERTO,28.633056,-17.755000,33.0,SANTA CRUZ DE TENERIFE,csv_metadata_fixed_parser,CAN_LP_LA_BAJITA,La Bajita,La Palma,Villa de Mazo,2.559014
2,C249I,FUERTEVENTURA AEROPUERTO,28.444722,-13.863056,25.0,LAS PALMAS,csv_metadata_fixed_parser,CAN_FV_PUERTO_ESCONDIDO,Puerto Escondido,Fuerteventura,Puerto del Rosario,1.041310
3,C329B,"LA GOMERA, AEROPUERTO",28.031667,-17.210833,219.0,STA. CRUZ DE TENERIFE,csv_metadata_fixed_parser,CAN_LG_SANTIAGO,Santiago,La Gomera,Alajeró,1.717875
4,C429I,TENERIFE SUR AEROPUERTO,28.046944,-16.561111,64.0,SANTA CRUZ DE TENERIFE,csv_metadata_fixed_parser,CAN_TF_LA_TEJITA,La Tejita,Tenerife,Granadilla de Abona,2.154591
5,C447A,TENERIFE NORTE AEROPUERTO,28.477500,-16.329444,632.0,SANTA CRUZ DE TENERIFE,csv_metadata_fixed_parser,CAN_TF_LA_NEA,La Nea,Tenerife,El Rosario,9.346274
6,C649I,GRAN CANARIA AEROPUERTO,27.917778,-15.395278,24.0,LAS PALMAS,csv_metadata_fixed_parser,CAN_GC_EL_BURRERO,El Burrero,Gran Canaria,Ingenio,1.419903
7,C929I,HIERRO AEROPUERTO,27.818889,-17.888889,32.0,SANTA CRUZ DE TENERIFE,csv_metadata_fixed_parser,CAN_EH_PUERTO_DE_LA_ESTACA,Puerto de la Estaca,El Hierro,Valverde,4.757155


Distancia estación AEMET → zona más cercana, km:


,distance_to_zona_km
count,8.000000
mean,3.040020
std,2.804206
min,1.041310
25%,1.395937
50%,1.936233
75%,3.108549
max,9.346274


## Celda 11 — Construir tabla Silver `meteo_hourly/source=AEMET`

In [13]:
aemet_silver = aemet_daily_raw.merge(
    station_zone[
        [
            "station_id",
            "station_name",
            "lat",
            "lon",
            "altitude_m",
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "distance_to_zona_km",
            "coordinate_source",
        ]
    ],
    on="station_id",
    how="left",
    suffixes=("", "_meta"),
)

# Si el JSON traía nombre, pero metadata no, usar el raw.
aemet_silver["station_name"] = aemet_silver["station_name"].fillna(aemet_silver["station_name_raw"])

aemet_silver["year"] = aemet_silver["timestamp"].dt.year.astype("Int64")

# Eliminar duplicados diarios.
before = len(aemet_silver)

aemet_silver = (
    aemet_silver
    .sort_values(["station_id", "timestamp", "source_file"])
    .drop_duplicates(subset=["station_id", "timestamp", "source"], keep="first")
    .copy()
)

duplicates_removed = before - len(aemet_silver)

print("Shape aemet_silver:", aemet_silver.shape)
print("Duplicados eliminados:", duplicates_removed)

if aemet_silver["zona_id"].isna().any():
    display(aemet_silver[aemet_silver["zona_id"].isna()].head())
    raise ValueError("Hay zona_id nulos tras asignar estaciones.")

if aemet_silver["lat"].isna().any() or aemet_silver["lon"].isna().any():
    raise ValueError("Hay lat/lon nulos tras asignar estaciones.")

display(aemet_silver.head())

Shape aemet_silver: (27578, 32)
Duplicados eliminados: 0


,timestamp,date,station_id,station_name_raw,temperature_air,temperature_min,temperature_max,precipitation,precipitation_is_trace,wind_speed,...,lat,lon,altitude_m,zona_id,nombre_zona,isla,municipio,distance_to_zona_km,coordinate_source,year
0,2000-01-01 00:00:00+00:00,2000-01-01,C029O,LANZAROTE AEROPUERTO,17.0,14.4,19.6,0.0,False,5.3,...,28.951944,-13.600278,14.0,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041,csv_metadata_fixed_parser,2000
1,2000-01-02 00:00:00+00:00,2000-01-02,C029O,LANZAROTE AEROPUERTO,16.2,13.5,19.0,0.0,False,4.4,...,28.951944,-13.600278,14.0,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041,csv_metadata_fixed_parser,2000
2,2000-01-03 00:00:00+00:00,2000-01-03,C029O,LANZAROTE AEROPUERTO,16.5,13.4,19.6,0.0,False,4.4,...,28.951944,-13.600278,14.0,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041,csv_metadata_fixed_parser,2000
3,2000-01-04 00:00:00+00:00,2000-01-04,C029O,LANZAROTE AEROPUERTO,16.6,13.8,19.4,0.0,False,3.6,...,28.951944,-13.600278,14.0,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041,csv_metadata_fixed_parser,2000
4,2000-01-05 00:00:00+00:00,2000-01-05,C029O,LANZAROTE AEROPUERTO,16.3,13.9,18.7,0.0,False,4.2,...,28.951944,-13.600278,14.0,CAN_LZ_HONDA,Honda,Lanzarote,San Bartolomé,1.324041,csv_metadata_fixed_parser,2000


## Celda 12 — Flags de calidad

In [14]:
VARIABLE_RANGES = {
    "wind_speed": (0, 60),
    "wind_direction": (0, 360),
    "wind_gust": (0, 80),
    "temperature_air": (-10, 50),
    "temperature_min": (-15, 50),
    "temperature_max": (-10, 55),
    "pressure": (800, 1100),
    "pressure_min": (800, 1100),
    "pressure_max": (800, 1100),
    "precipitation": (0, 500),
    "humidity": (0, 100),
    "u10": (-60, 60),
    "v10": (-60, 60),
}


def add_quality_flags(df, variable_ranges):
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df


aemet_silver = add_quality_flags(aemet_silver, VARIABLE_RANGES)

# Precipitación tipo Ip no es missing ni outlier: se conserva como 0.0.
if "precipitation_flag" in aemet_silver.columns:
    aemet_silver.loc[aemet_silver["precipitation_is_trace"].fillna(False), "precipitation_flag"] = 0

display(aemet_silver.head())

,timestamp,date,station_id,station_name_raw,temperature_air,temperature_min,temperature_max,precipitation,precipitation_is_trace,wind_speed,...,temperature_air_flag,temperature_min_flag,temperature_max_flag,pressure_flag,pressure_min_flag,pressure_max_flag,precipitation_flag,humidity_flag,u10_flag,v10_flag
0,2000-01-01 00:00:00+00:00,2000-01-01,C029O,LANZAROTE AEROPUERTO,17.0,14.4,19.6,0.0,False,5.3,...,0,0,0,0,0,0,0,0,0,0
1,2000-01-02 00:00:00+00:00,2000-01-02,C029O,LANZAROTE AEROPUERTO,16.2,13.5,19.0,0.0,False,4.4,...,0,0,0,0,0,0,0,0,0,0
2,2000-01-03 00:00:00+00:00,2000-01-03,C029O,LANZAROTE AEROPUERTO,16.5,13.4,19.6,0.0,False,4.4,...,0,0,0,0,0,0,0,0,0,0
3,2000-01-04 00:00:00+00:00,2000-01-04,C029O,LANZAROTE AEROPUERTO,16.6,13.8,19.4,0.0,False,3.6,...,0,0,0,0,0,0,0,0,0,0
4,2000-01-05 00:00:00+00:00,2000-01-05,C029O,LANZAROTE AEROPUERTO,16.3,13.9,18.7,0.0,False,4.2,...,0,0,0,0,0,0,0,0,0,0


## Celda 13 — Reportes de calidad y gaps

In [15]:
def gap_report_daily(df, station_col="station_id", timestamp_col="timestamp", expected_days=1):
    rows = []

    for station_id, g in df[[station_col, timestamp_col]].dropna().groupby(station_col):
        ts = g[timestamp_col].sort_values().drop_duplicates()
        diffs_d = ts.diff().dropna().dt.total_seconds() / 86400
        gaps = diffs_d[diffs_d > expected_days * 1.5]

        rows.append(
            {
                "station_id": station_id,
                "timestamp_min": ts.min(),
                "timestamp_max": ts.max(),
                "rows": len(ts),
                "gaps_count": int(len(gaps)),
                "max_gap_days": float(gaps.max()) if len(gaps) else 0.0,
                "expected_days": expected_days,
            }
        )

    return pd.DataFrame(rows)


quality_summary = pd.DataFrame(
    [
        {
            "table": "meteo_hourly",
            "source": SOURCE_NAME,
            "temporal_resolution": "daily",
            "rows": len(aemet_silver),
            "unique_stations": aemet_silver["station_id"].nunique(),
            "unique_zona_id": aemet_silver["zona_id"].nunique(),
            "timestamp_min": aemet_silver["timestamp"].min(),
            "timestamp_max": aemet_silver["timestamp"].max(),
            "duplicates_removed": duplicates_removed,
            "temperature_air_missing_pct": float(aemet_silver["temperature_air"].isna().mean() * 100),
            "precipitation_missing_pct": float(aemet_silver["precipitation"].isna().mean() * 100),
            "wind_speed_missing_pct": float(aemet_silver["wind_speed"].isna().mean() * 100),
            "wind_direction_missing_pct": float(aemet_silver["wind_direction"].isna().mean() * 100),
            "wind_gust_missing_pct": float(aemet_silver["wind_gust"].isna().mean() * 100),
            "pressure_missing_pct": float(aemet_silver["pressure"].isna().mean() * 100),
        }
    ]
)

missing_by_column = (
    aemet_silver.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

gaps_by_station = gap_report_daily(aemet_silver)

display(quality_summary)
display(missing_by_column)
display(gaps_by_station)

quality_summary.to_csv(QC_DIR / "quality_aemet_meteo_summary.csv", index=False)
missing_by_column.to_csv(QC_DIR / "quality_aemet_missing_by_column.csv", index=False)
gaps_by_station.to_csv(QC_DIR / "quality_aemet_gaps_by_station.csv", index=False)

,table,source,temporal_resolution,rows,unique_stations,unique_zona_id,timestamp_min,timestamp_max,duplicates_removed,temperature_air_missing_pct,precipitation_missing_pct,wind_speed_missing_pct,wind_direction_missing_pct,wind_gust_missing_pct,pressure_missing_pct
0,meteo_hourly,AEMET,daily,27578,8,8,2000-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,0,1.12771,0.0,0.670825,1.142215,1.142215,0.783233


,column,missing_pct
0,timestamp,0.000000
1,date,0.000000
2,station_id,0.000000
3,station_name_raw,0.000000
4,temperature_air,1.127710
5,temperature_min,1.105954
6,temperature_max,1.069693
7,precipitation,0.000000
8,precipitation_is_trace,0.000000
9,wind_speed,0.670825


,station_id,timestamp_min,timestamp_max,rows,gaps_count,max_gap_days,expected_days
0,C029O,2000-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4384,1,5114.0,1
1,C139E,2015-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4018,0,0.0,1
2,C249I,2015-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4018,0,0.0,1
3,C329B,2015-01-01 00:00:00+00:00,2017-12-31 00:00:00+00:00,1096,0,0.0,1
4,C429I,2015-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4018,0,0.0,1
5,C447A,2015-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4018,0,0.0,1
6,C649I,2015-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,4018,0,0.0,1
7,C929I,2015-01-01 00:00:00+00:00,2020-06-30 00:00:00+00:00,2008,0,0.0,1


## Celda 14 — Validaciones finales antes de guardar

In [16]:
required_cols = [
    "timestamp",
    "station_id",
    "station_name",
    "zona_id",
    "lat",
    "lon",
    "source",
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "temperature_min",
    "temperature_max",
    "pressure",
    "pressure_min",
    "pressure_max",
    "precipitation",
    "humidity",
    "u10",
    "v10",
    "temporal_resolution",
    "year",
    "isla",
]

missing_required = [c for c in required_cols if c not in aemet_silver.columns]

if missing_required:
    raise ValueError(f"Faltan columnas requeridas: {missing_required}")

if aemet_silver.empty:
    raise ValueError("aemet_silver está vacío.")

if aemet_silver["timestamp"].isna().any():
    raise ValueError("Hay timestamps nulos.")

if aemet_silver["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos.")

if aemet_silver["lat"].isna().any() or aemet_silver["lon"].isna().any():
    raise ValueError("Hay coordenadas nulas.")

invalid_time = ~aemet_silver["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

if invalid_time.any():
    raise ValueError(
        "Hay timestamps fuera de rango: "
        f"{aemet_silver.loc[invalid_time, 'timestamp'].min()} - "
        f"{aemet_silver.loc[invalid_time, 'timestamp'].max()}"
    )

# Variables clave reales diarias.
if aemet_silver["temperature_air"].isna().mean() > 0.25:
    raise ValueError("Más del 25% de temperature_air está nulo. Revisar mapeo.")

if aemet_silver["precipitation"].isna().mean() > 0.25:
    raise ValueError("Más del 25% de precipitation está nulo. Revisar mapeo.")

# Viento/presión pueden tener más huecos según estación, avisar.
if aemet_silver["wind_speed"].isna().mean() > 0.5:
    print("AVISO: más del 50% de wind_speed está nulo en AEMET.")

if aemet_silver["pressure"].isna().mean() > 0.5:
    print("AVISO: más del 50% de pressure está nulo en AEMET.")

print("Validaciones finales AEMET superadas.")

Validaciones finales AEMET superadas.


## Celda 15 — Guardar Parquet particionado

In [17]:
final_cols = [
    "timestamp",
    "date",
    "station_id",
    "station_name",
    "zona_id",
    "lat",
    "lon",
    "altitude_m",
    "source",
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "temperature_min",
    "temperature_max",
    "pressure",
    "pressure_min",
    "pressure_max",
    "precipitation",
    "precipitation_is_trace",
    "humidity",
    "u10",
    "v10",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
    "municipio",
    "coordinate_source",
    "source_file",
]

# Añadir flags existentes.
flag_cols = [c for c in aemet_silver.columns if c.endswith("_flag")]
final_cols = final_cols + flag_cols

for col in final_cols:
    if col not in aemet_silver.columns:
        aemet_silver[col] = np.nan

aemet_final = aemet_silver[final_cols].copy()

remove_existing_source_partition(OUT_METEO_DIR, SOURCE_NAME)
write_partitioned_parquet(aemet_final, OUT_METEO_DIR)

print("Guardado AEMET en:")
print(OUT_METEO_DIR / f"source={SOURCE_NAME}")

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=AEMET
Guardado AEMET en:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/meteo_hourly/source=AEMET


## Celda 16 — Comprobación final de lectura

In [18]:
aemet_count, aemet_sample = dataset_count_and_sample(OUT_METEO_DIR)

print("Filas guardadas meteo_hourly AEMET:", aemet_count)

if len(aemet_sample):
    display(aemet_sample)

if aemet_count == 0:
    raise ValueError("No se guardó ningún registro AEMET.")

global_summary = pd.DataFrame(
    [
        {
            "table": "meteo_hourly",
            "source": SOURCE_NAME,
            "rows": aemet_count,
            "stations": aemet_final["station_id"].nunique(),
            "timestamp_min": aemet_final["timestamp"].min(),
            "timestamp_max": aemet_final["timestamp"].max(),
            "temperature_air_missing_pct": float(aemet_final["temperature_air"].isna().mean() * 100),
            "precipitation_missing_pct": float(aemet_final["precipitation"].isna().mean() * 100),
            "wind_speed_missing_pct": float(aemet_final["wind_speed"].isna().mean() * 100),
            "pressure_missing_pct": float(aemet_final["pressure"].isna().mean() * 100),
        }
    ]
)

display(global_summary)

global_summary.to_csv(QC_DIR / "quality_aemet_global_summary.csv", index=False)

print("Reportes AEMET:")
for p in sorted(QC_DIR.glob("quality_aemet*.csv")):
    print("-", p)

print("\nMetadatos AEMET:")
for p in sorted(META_DIR.glob("aemet*.csv")):
    print("-", p)

print("\nValidación final AEMET superada.")

Filas guardadas meteo_hourly AEMET: 27578


,timestamp,date,station_id,station_name,zona_id,lat,lon,altitude_m,wind_speed,wind_direction,...,pressure_flag,pressure_min_flag,pressure_max_flag,precipitation_flag,humidity_flag,u10_flag,v10_flag,source,year,isla
0,2000-01-01 00:00:00+00:00,2000-01-01,C029O,LANZAROTE AEROPUERTO,CAN_LZ_HONDA,28.951944,-13.600278,14.0,5.3,60.0,...,0,0,0,0,0,0,0,AEMET,2000,Lanzarote
1,2000-01-02 00:00:00+00:00,2000-01-02,C029O,LANZAROTE AEROPUERTO,CAN_LZ_HONDA,28.951944,-13.600278,14.0,4.4,120.0,...,0,0,0,0,0,0,0,AEMET,2000,Lanzarote
2,2000-01-03 00:00:00+00:00,2000-01-03,C029O,LANZAROTE AEROPUERTO,CAN_LZ_HONDA,28.951944,-13.600278,14.0,4.4,70.0,...,0,0,0,0,0,0,0,AEMET,2000,Lanzarote
3,2000-01-04 00:00:00+00:00,2000-01-04,C029O,LANZAROTE AEROPUERTO,CAN_LZ_HONDA,28.951944,-13.600278,14.0,3.6,20.0,...,0,0,0,0,0,0,0,AEMET,2000,Lanzarote
4,2000-01-05 00:00:00+00:00,2000-01-05,C029O,LANZAROTE AEROPUERTO,CAN_LZ_HONDA,28.951944,-13.600278,14.0,4.2,170.0,...,0,0,0,0,0,0,0,AEMET,2000,Lanzarote


,table,source,rows,stations,timestamp_min,timestamp_max,temperature_air_missing_pct,precipitation_missing_pct,wind_speed_missing_pct,pressure_missing_pct
0,meteo_hourly,AEMET,27578,8,2000-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.12771,0.0,0.670825,0.783233


Reportes AEMET:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_gaps_by_station.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_meteo_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_missing_by_column.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_aemet_read_errors.csv

Metadatos AEMET:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/aemet_station_to_zone.csv

Validación final AEMET superada.


## Resultado esperado

Al terminar deberían existir:

```text
silver/meteo_hourly/source=AEMET/year=YYYY/isla=.../*.parquet
silver/_quality_reports/quality_aemet_global_summary.csv
silver/_quality_reports/quality_aemet_meteo_summary.csv
silver/_quality_reports/quality_aemet_missing_by_column.csv
silver/_quality_reports/quality_aemet_gaps_by_station.csv
silver/_metadata/aemet_station_to_zone.csv
```

Comprueba especialmente:

```text
Errores de lectura = 0
Filas guardadas meteo_hourly AEMET > 0
temperature_air_missing_pct razonable
precipitation_missing_pct razonable
Validación final AEMET superada
```

AEMET queda como `temporal_resolution = daily`; no se interpola a horario en Silver.